# Notebook 04 — Image Upsampling with Nearest Neighbor Interpolation

**Session:** Interpolation and Image upsampling  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement Nearest Neighbor Interpolation (NNI) image resizing in two ways—a two-step
separable horizontal-then-vertical pass and an equivalent one-shot coordinate mapping—verify
their equivalence on a $3 \times 3$ toy matrix, and apply $2\times$ upsampling to
`../data/nature.jpg`.

In [ ]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGE_PATH = Path("../data/nature.jpg")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Two-Step Separable vs. One-Shot Nearest Neighbor Resizing

In [ ]:
def resize_nni_two_step(
    source: list[list[int]],
    target_h: int,
    target_w: int,
) -> list[list[int]]:
    """Resize 2D grid via horizontal NNI followed by vertical NNI."""
    source_h, source_w = len(source), len(source[0])
    w_scale = source_w / target_w
    h_scale = source_h / target_h

    horiz = [[0] * target_w for _ in range(source_h)]
    for i in range(source_h):
        for j in range(target_w):
            horiz[i][j] = source[i][math.floor(j * w_scale)]

    out = [[0] * target_w for _ in range(target_h)]
    for j in range(target_w):
        for i in range(target_h):
            out[i][j] = horiz[math.floor(i * h_scale)][j]
    return out


def resize_nni_one_shot(
    source: list[list[int]],
    target_h: int,
    target_w: int,
) -> list[list[int]]:
    """Resize 2D grid in a single coordinate-mapping pass."""
    source_h, source_w = len(source), len(source[0])
    w_scale = source_w / target_w
    h_scale = source_h / target_h

    out = [[0] * target_w for _ in range(target_h)]
    for i in range(target_h):
        y = math.floor(i * h_scale)
        for j in range(target_w):
            x = math.floor(j * w_scale)
            out[i][j] = source[y][x]
    return out


toy = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
res_two = resize_nni_two_step(toy, 6, 6)
res_one = resize_nni_one_shot(toy, 6, 6)
assert res_two == res_one
print("2x NNI on 3x3 toy grid:")
for row in res_one:
    print(" ", row)

## 2. 2x Upsampling on a Real Grayscale Image

In [ ]:
image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE).tolist()
h, w = len(image), len(image[0])

upsampled_nni = resize_nni_one_shot(image, h * 2, w * 2)
upsampled_arr = np.asarray(upsampled_nni, dtype=np.uint8)
cv2.imwrite(str(OUTPUT_DIR / "04_nature_nni_2x.jpg"), upsampled_arr)

print(f"Original: ({h}, {w}) -> Upsampled NNI: {upsampled_arr.shape}")
plt.figure(figsize=(8, 4))
plt.imshow(upsampled_arr, cmap="gray")
plt.title(f"2x Nearest Neighbor Upsampling {upsampled_arr.shape}")
plt.axis("off")
plt.tight_layout()
plt.show()